In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-13_07-31-25-109_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-13_07-06-14-938_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-11_07-18-20-122_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-11_08-03-55-015_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-11_07-55-12-414_textready_cleaned.xlsx

In [1]:
# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout, Bidirectional,
    SpatialDropout1D, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/youtube_sentiment_cleaned_3class"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

MAX_WORDS = 30000
MAX_LEN = 80
EMBED_DIM = 96

BATCH_SIZE = 96
EPOCHS = 15
LEARNING_RATE = 3e-4

USE_CLASS_WEIGHTS = True
BOOST_NEGATIVE_WEIGHT = True
NEGATIVE_BOOST_FACTOR = 1.08

DOWNSAMPLE_NEUTRAL = True
NEUTRAL_MAX = 15000

TEXT_COLS = ["Text_TR", "Text_ML", "Text_Cleaned", "Text", "Text_Orig"]

# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================
# 4) LOAD CLEANED FILES
# =========================================
root = Path(ROOT_FOLDER)
cleaned_files = sorted(root.rglob("*_textready_cleaned.xlsx"))

print("Cleaned files:", len(cleaned_files))

dfs = []
bad_files = []

for f in cleaned_files:
    try:
        df_part = pd.read_excel(f)
        df_part["Source_File"] = f.name
        df_part["Parent_Folder"] = f.parent.name
        dfs.append(df_part)
    except Exception as e:
        bad_files.append((str(f), str(e)))

if not dfs:
    raise ValueError("No cleaned files could be loaded.")

df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", df.shape)

if bad_files:
    pd.DataFrame(bad_files, columns=["file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_cleaned_files.xlsx"), index=False
    )

# =========================================
# 5) CHOOSE TEXT COLUMN
# =========================================
text_col = None
for col in TEXT_COLS:
    if col in df.columns:
        non_empty = df[col].notna().sum()
        if non_empty > 0:
            text_col = col
            break

if text_col is None:
    raise ValueError("No suitable text column found.")

print("Using text column:", text_col)

df[text_col] = df[text_col].fillna("").astype(str).str.strip()
df = df[df[text_col] != ""].copy()

# remove ultra-short texts
df["word_count"] = df[text_col].astype(str).str.split().str.len()
df = df[df["word_count"] >= 2].copy()

# =========================================
# 6) REMOVE UNWANTED COMMENT TYPES
# =========================================
if "comment_type" in df.columns:
    df["comment_type"] = df["comment_type"].astype(str).str.strip().str.lower()
    df = df[~df["comment_type"].isin(["ad", "noise", "channel_comment"])].copy()

print("After removing ad/noise/channel_comment:", df.shape)

# =========================================
# 7) BUILD 3-CLASS LABEL
# =========================================
def build_label(row):
    if "Sentiment" in row and pd.notna(row["Sentiment"]):
        s = str(row["Sentiment"]).strip().lower()
        if s in ["positive", "pos", "1"]:
            return "positive"
        elif s in ["negative", "neg", "-1"]:
            return "negative"
        elif s in ["neutral", "neu", "0"]:
            return "neutral"

    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            star = float(row["Stars"])
            if star >= 4:
                return "positive"
            elif star <= 2:
                return "negative"
            else:
                return "neutral"
        except:
            return None

    return None

df["label"] = df.apply(build_label, axis=1)
df = df.dropna(subset=["label"]).copy()

print("\nLabel distribution before balancing:")
print(df["label"].value_counts())

# =========================================
# 8) DOWNSAMPLE NEUTRAL
# =========================================
if DOWNSAMPLE_NEUTRAL:
    neutral_df = df[df["label"] == "neutral"].copy()
    positive_df = df[df["label"] == "positive"].copy()
    negative_df = df[df["label"] == "negative"].copy()

    if len(neutral_df) > NEUTRAL_MAX:
        neutral_df = neutral_df.sample(n=NEUTRAL_MAX, random_state=RANDOM_STATE)

    df = pd.concat([positive_df, negative_df, neutral_df], ignore_index=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nLabel distribution after balancing:")
print(df["label"].value_counts())

# =========================================
# 9) ENCODE LABELS
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")

# =========================================
# 10) SPLIT
# =========================================
X = df[text_col].astype(str).tolist()
y = df["y"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, df.index.values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nTrain size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================
# 11) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

# =========================================
# 12) CLASS WEIGHTS
# =========================================
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    if BOOST_NEGATIVE_WEIGHT:
        negative_id = int(le.transform(["negative"])[0])
        class_weight_dict[negative_id] *= NEGATIVE_BOOST_FACTOR

    print("\nClass weights:")
    print(class_weight_dict)

# =========================================
# 13) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),
    SpatialDropout1D(0.30),

    Bidirectional(
        LSTM(
            48,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    Bidirectional(
        LSTM(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    GlobalMaxPooling1D(),

    Dense(48, activation="relu", kernel_regularizer=l2(8e-5)),
    Dropout(0.45),

    Dense(num_classes, activation="softmax")
])

optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()

# =========================================
# 14) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_3class.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================
# 15) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================
# 16) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

# =========================================
# 17) SAVE OUTPUTS
# =========================================
report_dict = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)

test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)

model.save(os.path.join(OUTPUT_DIR, "final_3class.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(OUTPUT_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "text_col": text_col,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "BOOST_NEGATIVE_WEIGHT": BOOST_NEGATIVE_WEIGHT,
    "NEGATIVE_BOOST_FACTOR": NEGATIVE_BOOST_FACTOR,
    "DOWNSAMPLE_NEUTRAL": DOWNSAMPLE_NEUTRAL,
    "NEUTRAL_MAX": NEUTRAL_MAX,
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "label_distribution": {str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()}
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

2026-04-15 18:06:01.245372: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776276361.495144      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776276361.564366      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776276362.089761      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776276362.089806      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776276362.089809      55 computation_placer.cc:177] computation placer alr

Cleaned files: 198
Initial shape: (58175, 36)
Using text column: Text_TR
After removing ad/noise/channel_comment: (44926, 37)

Label distribution before balancing:
label
positive    20733
negative    12498
neutral     11695
Name: count, dtype: int64

Label distribution after balancing:
label
positive    20733
negative    12498
neutral     11695
Name: count, dtype: int64

Encoded classes:
0 -> negative
1 -> neutral
2 -> positive

Train size: 35940
Val size: 4493
Test size: 4493

Class weights:
{0: 1.2940988197639527, 1: 1.2804617357845234, 2: 0.7222959122151212}


I0000 00:00:1776276417.686710      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776276417.692695      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 728ms/step - accuracy: 0.3748 - loss: 1.1401
Epoch 1: val_loss improved from inf to 0.71257, saving model to /kaggle/working/youtube_sentiment_cleaned_3class/best_3class.keras
375/375 ━━━━━━━━━━━━━━━━━━━━ 297s 748ms/step - accuracy: 0.3750 - loss: 1.1399 - val_accuracy: 0.7133 - val_loss: 0.7126 - learning_rate: 3.0000e-04
Epoch 2/15
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 724ms/step - accuracy: 0.7182 - loss: 0.7349
Epoch 2: val_loss improved from 0.71257 to 0.55311, saving model to /kaggle/working/youtube_sentiment_cleaned_3class/best_3class.keras
375/375 ━━━━━━━━━━━━━━━━━━━━ 278s 741ms/step - accuracy: 0.7183 - loss: 0.7347 - val_accuracy: 0.7799 - val_loss: 0.5531 - learning_rate: 3.0000e-04
Epoch 3/15
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 730ms/step - accuracy: 0.8194 - loss: 0.5256
Epoch 3: val_loss improved from 0.55311 to 0.52225, saving model to /kaggle/working/youtube_sentiment_cleaned_3class/best_3class.keras
375/375 ━━━━━━━━━━━━━━━━━━━━ 280s 747

In [1]:
# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout, Bidirectional,
    SpatialDropout1D, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/youtube_sentiment_binary"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

TEXT_COL = "Text_TR"

MAX_WORDS = 30000
MAX_LEN = 80
EMBED_DIM = 96

BATCH_SIZE = 96
EPOCHS = 15
LEARNING_RATE = 3e-4

USE_CLASS_WEIGHTS = True
BOOST_NEGATIVE_WEIGHT = True
NEGATIVE_BOOST_FACTOR = 1.10

REMOVE_EMPTY_ONLY = True

EXCLUDED_COMMENT_TYPES = {
    "ad",
    "ads",
    "noise",
    "noisy",
    "question",
    "questions",
    "channel_comment"
}

# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================
# 4) LOAD CLEANED FILES
# =========================================
root = Path(ROOT_FOLDER)
cleaned_files = sorted(root.rglob("*_textready_cleaned.xlsx"))

print("Cleaned files:", len(cleaned_files))

dfs = []
bad_files = []

for f in cleaned_files:
    try:
        df_part = pd.read_excel(f)
        df_part["Source_File"] = f.name
        df_part["Parent_Folder"] = f.parent.name
        dfs.append(df_part)
    except Exception as e:
        bad_files.append((str(f), str(e)))

if not dfs:
    raise ValueError("No cleaned files could be loaded.")

df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", df.shape)

if bad_files:
    pd.DataFrame(bad_files, columns=["file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_cleaned_files.xlsx"), index=False
    )

# =========================================
# 5) KEEP ONLY Text_TR
# =========================================
if TEXT_COL not in df.columns:
    raise ValueError(f"{TEXT_COL} column was not found.")

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()
df[TEXT_COL] = df[TEXT_COL].str.replace(r"\s+", " ", regex=True)

before_text = len(df)
if REMOVE_EMPTY_ONLY:
    df = df[df[TEXT_COL] != ""].copy()
after_text = len(df)

print(f"After Text_TR non-empty filtering: {before_text} -> {after_text}")

# =========================================
# 6) REMOVE UNWANTED COMMENT TYPES
# =========================================
if "comment_type" in df.columns:
    df["comment_type"] = df["comment_type"].fillna("").astype(str).str.strip().str.lower()
    df = df[~df["comment_type"].isin(EXCLUDED_COMMENT_TYPES)].copy()

print("After removing excluded comment types:", df.shape)

# =========================================
# 7) BUILD BINARY LABEL FROM Stars ONLY
# positive / negative فقط
# =========================================
def build_binary_label(row):
    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            star = float(row["Stars"])
            if star >= 4:
                return "positive"
            elif star <= 2:
                return "negative"
            else:
                return None   # حذف المحايد
        except:
            return None
    return None

df["label"] = df.apply(build_binary_label, axis=1)
df = df.dropna(subset=["label"]).copy()

print("\nBinary label distribution:")
print(df["label"].value_counts())

# =========================================
# 8) ENCODE LABELS
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")

# =========================================
# 9) SPLIT
# =========================================
X = df[TEXT_COL].astype(str).tolist()
y = df["y"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, df.index.values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nTrain size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================
# 10) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

# =========================================
# 11) CLASS WEIGHTS
# =========================================
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    if BOOST_NEGATIVE_WEIGHT and "negative" in le.classes_:
        negative_id = int(le.transform(["negative"])[0])
        class_weight_dict[negative_id] *= NEGATIVE_BOOST_FACTOR

    print("\nClass weights:")
    print(class_weight_dict)

# =========================================
# 12) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),
    SpatialDropout1D(0.30),

    Bidirectional(
        LSTM(
            48,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    Bidirectional(
        LSTM(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    GlobalMaxPooling1D(),

    Dense(48, activation="relu", kernel_regularizer=l2(8e-5)),
    Dropout(0.45),

    Dense(num_classes, activation="softmax")
])

optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()

# =========================================
# 13) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_binary.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================
# 14) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================
# 15) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

# =========================================
# 16) SAVE OUTPUTS
# =========================================
report_dict = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)

test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)

model.save(os.path.join(OUTPUT_DIR, "final_binary.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(OUTPUT_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "TEXT_COL": TEXT_COL,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "BOOST_NEGATIVE_WEIGHT": BOOST_NEGATIVE_WEIGHT,
    "NEGATIVE_BOOST_FACTOR": NEGATIVE_BOOST_FACTOR,
    "REMOVE_EMPTY_ONLY": REMOVE_EMPTY_ONLY,
    "EXCLUDED_COMMENT_TYPES": sorted(list(EXCLUDED_COMMENT_TYPES)),
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "label_distribution": {str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()}
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

2026-04-18 13:10:24.341773: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776517824.567992      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776517824.627675      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776517825.123009      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776517825.123058      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776517825.123062      55 computation_placer.cc:177] computation placer alr

Cleaned files: 198
Initial shape: (58175, 36)
After Text_TR non-empty filtering: 58175 -> 58175
After removing excluded comment types: (37568, 36)

Binary label distribution:
label
positive    19589
negative    11973
Name: count, dtype: int64

Encoded classes:
0 -> negative
1 -> positive

Train size: 25249
Val size: 3156
Test size: 3157

Class weights:
{0: 1.4498799331802046, 1: 0.8055963244209049}


I0000 00:00:1776517885.038554      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776517885.044835      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 833ms/step - accuracy: 0.5241 - loss: 0.7345
Epoch 1: val_loss improved from inf to 0.46179, saving model to /kaggle/working/youtube_sentiment_binary/best_binary.keras
264/264 ━━━━━━━━━━━━━━━━━━━━ 246s 859ms/step - accuracy: 0.5246 - loss: 0.7342 - val_accuracy: 0.8150 - val_loss: 0.4618 - learning_rate: 3.0000e-04
Epoch 2/15
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 835ms/step - accuracy: 0.8430 - loss: 0.4181
Epoch 2: val_loss improved from 0.46179 to 0.37584, saving model to /kaggle/working/youtube_sentiment_binary/best_binary.keras
264/264 ━━━━━━━━━━━━━━━━━━━━ 226s 855ms/step - accuracy: 0.8431 - loss: 0.4179 - val_accuracy: 0.8606 - val_loss: 0.3758 - learning_rate: 3.0000e-04
Epoch 3/15
264/264 ━━━━━━━━━━━━━━━━━━━━ 0s 838ms/step - accuracy: 0.9051 - loss: 0.2864
Epoch 3: val_loss improved from 0.37584 to 0.37349, saving model to /kaggle/working/youtube_sentiment_binary/best_binary.keras
264/264 ━━━━━━━━━━━━━━━━━━━━ 226s 857ms/step - accuracy: 0.90

In [2]:
# =========================================
# SAVE YOUTUBE BINARY MODEL
# =========================================
import os
import json
import pickle
import shutil

SAVE_DIR = OUTPUT_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) Save final model
model.save(os.path.join(SAVE_DIR, "final_binary.keras"))

# 2) Copy best model (الأفضل)
best_src = os.path.join(SAVE_DIR, "best_binary.keras")
best_dst = os.path.join(SAVE_DIR, "approved_best_binary.keras")

if os.path.exists(best_src):
    shutil.copy(best_src, best_dst)
    print("Best model copied to:", best_dst)
else:
    print("best_binary.keras not found")

# 3) Save tokenizer
with open(os.path.join(SAVE_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

# 4) Save label encoder
with open(os.path.join(SAVE_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

# 5) Save config (مهم للتشغيل لاحقًا)
config = {
    "TEXT_COL": TEXT_COL,
    "classes": list(le.classes_),
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE
}

with open(os.path.join(SAVE_DIR, "model_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nSaved successfully:")
print(os.path.join(SAVE_DIR, "approved_best_binary.keras"))
print(os.path.join(SAVE_DIR, "tokenizer.pkl"))
print(os.path.join(SAVE_DIR, "label_encoder.pkl"))
print(os.path.join(SAVE_DIR, "model_config.json"))

Best model copied to: /kaggle/working/youtube_sentiment_binary/approved_best_binary.keras

Saved successfully:
/kaggle/working/youtube_sentiment_binary/approved_best_binary.keras
/kaggle/working/youtube_sentiment_binary/tokenizer.pkl
/kaggle/working/youtube_sentiment_binary/label_encoder.pkl
/kaggle/working/youtube_sentiment_binary/model_config.json
